# 03 Feature Engineering

This notebook focuses on the feature engineering pipeline for the Sentinel Sepsis-Associated AKI (SA-AKI) Early Warning System.

## Strategy
- **Missingness Indicators**: Missing data in EHR is highly informative. We explicitly create missingness flags before imputation.
- **Rolling Windows**: To capture trends in vital signs and lab results, we compute rolling aggregates (mean, max, min, std) over time windows (e.g., 6h, 12h, 24h).
- **Creatinine Velocity**: The rate of change in serum creatinine is a critical physiological marker for acute kidney injury.
- **Composite Features**: Combining related measurements (e.g., BUN-to-creatinine ratio, shock index).

In [2]:
# Colab setup: Uncomment if running in Google Colab
!pip install -q pandas numpy
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/sentinel-poc/project_sentinel

Mounted at /content/drive
/content/drive/MyDrive/sentinel-poc/project_sentinel


In [4]:
import os, sys
# point this at YOUR clone location if different
PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
os.chdir(os.path.join(PROJ, 'notebooks'))   # so ../data/... paths resolve
sys.path.insert(0, PROJ)                      # so `from src...` works
print('cwd =', os.getcwd())

cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys, os
sys.path.append(os.path.abspath('..'))

from src import features

interim_dir = Path('../data/interim')

print("Loading defined cohort...")
cohort_df = pd.read_parquet(interim_dir / 'cohort_defined.parquet')
print(f"Loaded {cohort_df['patient_id'].nunique()} stays, {len(cohort_df):,} rows.")

print("Loading baseline creatinine...")
baseline_cr = pd.read_parquet(interim_dir / 'baseline_creatinine.parquet')
print(f"Loaded {len(baseline_cr)} baselines.")

Loading defined cohort...
Loaded 136 stays, 12,413 rows.
Loading baseline creatinine...
Loaded 136 baselines.


In [6]:
print("Adding missingness indicators...")
df_with_missingness = features.add_missingness_indicators(cohort_df)
print(f"Dataset shape after missingness indicators: {df_with_missingness.shape}")

Adding missingness indicators...
Dataset shape after missingness indicators: (12413, 65)


In [7]:
print("Adding rolling features...")
df_rolling = features.add_rolling_features(df_with_missingness)
print(f"Dataset shape after rolling features: {df_rolling.shape}")

print("Imputing missing values...")
df_imputed = features.impute_values(df_rolling)
print(f"Dataset shape after imputation: {df_imputed.shape}")

Adding rolling features...
Dataset shape after rolling features: (12413, 193)
Imputing missing values...
Dataset shape after imputation: (12413, 193)


## Creatinine Velocity

Creatinine velocity is perhaps the most important feature for predicting SA-AKI. It captures the **rate of change** of serum creatinine over recent time windows (e.g., 24 hours, 48 hours).

A rapid rise in creatinine is a stronger predictor of imminent kidney injury than the absolute level of creatinine alone, as it directly reflects acute loss of glomerular filtration.

In [8]:
print("Calculating creatinine velocity...")
df_cr_velocity = features.add_creatinine_velocity(df_imputed, baseline_cr)
print(f"Dataset shape after creatinine velocity: {df_cr_velocity.shape}")

Calculating creatinine velocity...
Dataset shape after creatinine velocity: (12413, 197)


In [9]:
print("Adding composite features...")
df_composites = features.add_composite_features(df_cr_velocity)
print(f"Dataset shape after composite features: {df_composites.shape}")

Adding composite features...
Dataset shape after composite features: (12413, 203)


In [10]:
print("Running the full feature-engineering pipeline...")
# Cells 3–7 above illustrate each step; engineer_features runs them in the
# correct order (missingness → rolling on sparse data → impute → velocity → composites).
final_features_df = features.engineer_features(cohort_df, baseline_cr)
print(f"Final dataset shape: {final_features_df.shape}")

features_path = interim_dir / 'engineered_features.parquet'
final_features_df.to_parquet(features_path, index=False)
print(f"Saved → {features_path}")

Running the full feature-engineering pipeline...
Final dataset shape: (12413, 203)
Saved → ../data/interim/engineered_features.parquet


In [11]:
print("Creating feature manifest...")
manifest = features.create_feature_manifest(
    final_features_df, interim_dir / 'feature_manifest.csv'
)
print(f"Generated manifest with {len(manifest)} features → {interim_dir / 'feature_manifest.csv'}")
display(manifest.head(15))

Creating feature manifest...
Generated manifest with 198 features → ../data/interim/feature_manifest.csv


,name,category,pct_missing,mean,std,min,max
0,stay_id,other,0.00,3.478872e+07,2.902747e+06,30057454.00,39880770.0
1,DBP,vital,3.65,6.210180e+01,1.326590e+01,18.00,162.0
2,HR,vital,1.04,9.060610e+01,1.842540e+01,21.25,163.0
3,MAP,vital,1.39,7.682650e+01,1.633540e+01,8.00,801.0
4,O2Sat,vital,1.34,9.670760e+01,3.000200e+00,29.00,100.0
5,Resp,vital,0.75,2.004380e+01,5.414500e+00,0.00,58.0
6,SBP,vital,3.65,1.159939e+02,2.020930e+01,46.00,207.0
7,Temp,vital,6.27,3.701480e+01,1.964400e+00,31.30,99.0
8,AST,lab,82.41,3.607840e+02,1.224057e+03,3.00,13060.0
9,Alkalinephos,lab,82.41,1.172446e+02,7.446570e+01,28.00,515.0


## Next Step

Engineered features are saved to `data/interim/engineered_features.parquet` (raw clinical columns are retained alongside the derived features, so labeling can still read `Creatinine`, `urine_rate`, etc.).

The chronological **train/val/test split happens in `04_label_engineering.ipynb`** — *after* the KDIGO labels are attached — so each split carries features **and** labels together with no leakage.

In [12]:
!git config --global user.email "dcsenga442@gmail.com"
!git config --global user.name  "Sng43"

from getpass import getpass
tok = getpass("GitHub PAT: ")
# We use the token to set the remote, then immediately clear it from memory
!cd /content/drive/MyDrive/sentinel-poc && git remote set-url origin https://{tok}@github.com/Sng43/sentinel-poc.git

# Clear the variable and the cell output manually to ensure it isn't saved in the .ipynb
tok = None
print("Remote updated. Token cleared from memory.")

GitHub PAT: ··········
Remote updated. Token cleared from memory.


In [13]:
%cd /content/drive/MyDrive/sentinel-poc
!git add -A
!git commit -m "run: 01_data_acquisition (clean)"
!git push

/content/drive/MyDrive/sentinel-poc
[main a944d40] run: 03_feature_engineering.ipynb (clean)
 2 files changed, 2 insertions(+), 131 deletions(-)
 rewrite project_sentinel/notebooks/03_feature_engineering.ipynb (98%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 7.33 KiB | 750.00 KiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Sng43/sentinel-poc.git
   8781486..a944d40  main -> main
